# WarBias evaluation after intervention

Compare each saved WGA+KL or scale-one task-vector checkpoint with its own matched base.
The [experiment notebook](warbias_unlearning.ipynb) describes the 137-triplet Ukrainian
training bank and both methods.

This notebook reads verified Fair-UK results. Metric values and confidence intervals
are loaded unchanged; notebook cells only select and display them. No new model inference
runs.

## Load the saved experiment

Use Python 3.10 or later with `pandas`, `matplotlib`, `nbformat` and `nbclient`.
Run from this checkout or its `notebooks/` directory.
The shared loader verifies archive hashes, source files, training receipts and matched pairs.
It reads the WGA archive already in the repository and downloads the pinned task-vector
archive (18 MB) on first use. Extracted evidence uses about 1.3 GB under
`outputs/warbias_experiment/`. Set `WARBIAS_EXPERIMENT_CACHE` to use another location.
No model weights or credentials are needed.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "scripts/warbias_experiment_data.py").is_file()
)
sys.path.insert(0, str(repo / "scripts"))
from warbias_experiment_data import load_experiment, qa_rows  # noqa: E402

reports = load_experiment(repo, cache=os.environ.get("WARBIAS_EXPERIMENT_CACHE"))
labels = {"wga_kl": "WGA+KL", "task_vector": "Task vector (scale 1)"}


def show(frame):
    display(
        HTML(
            '<div dir="ltr" style="text-align:left">'
            + frame.to_html(index=False, justify="left", float_format=lambda x: f"{x:.6g}")
            + "</div><style>table.dataframe th, table.dataframe td {text-align:left}</style>"
        )
    )


print("Verified saved results: two methods, six models. No new training or inference.")

Verified saved results: two methods, six models. No new training or inference.


## Evaluation protocol

Each method has 72 held-out pairs: six models and twelve Ukrainian/English views.
Triplets use full-sentence mean token log-probabilities. QA uses the recorded `abc_option_text_v2` answer policy. Each pair preserves its source inputs, model lineage and runtime settings.
Mamay 27B task-vector pairs use the recorded batch-invariant runtime with TRITON_ATTN.

The recorded pairs are separate from the final-release prior-normalised baseline in
[notebook 08](../warbias-artifacts/notebooks/08_warbias_baseline.ipynb).
Do not combine scores from different dataset versions or scoring protocols.

In [2]:
coverage = pd.concat(
    [
        pd.DataFrame(report["warbias_task_coverage"]).assign(method=labels[method])
        for method, report in reports.items()
    ],
    ignore_index=True,
)
show(coverage.pivot(index=["method", "task"], columns="model", values="status").reset_index())
show(
    pd.DataFrame(
        [
            {
                "method": labels[method],
                "manifest_sha256": report["manifest_sha256"],
                "source_revision": report["training_scope"]["forget_revision"],
            }
            for method, report in reports.items()
        ]
    )
)

model,method,task,Gemma 3 12B IT,Lapa 12B Instruct,Llama 3.1 8B Instruct,Mamay 12B v2,Mamay 27B v2,Qwen3 8B
,Task vector (scale 1),warbias_benign_en,available,available,available,available,available,available
,Task vector (scale 1),warbias_benign_uk,available,available,available,available,available,available
,Task vector (scale 1),warbias_cross_actor_en,available,available,available,available,available,available
,Task vector (scale 1),warbias_cross_actor_uk,available,available,available,available,available,available
,Task vector (scale 1),warbias_en,available,available,available,available,available,available
,Task vector (scale 1),warbias_expanded_qa_en,available,available,available,available,available,available
,Task vector (scale 1),warbias_expanded_qa_uk,available,available,available,available,available,available
,Task vector (scale 1),warbias_intersectional_en,available,available,available,available,available,available
,Task vector (scale 1),warbias_intersectional_uk,available,available,available,available,available,available
,Task vector (scale 1),warbias_triplets_en,available,available,available,available,available,available


method,manifest_sha256,source_revision
WGA+KL,ea7cb1ee26c8b7df86910d8041308e9d61b2caeb3a037cf6cd51d7e66bdfafae,164f355623c2da0028821e0942cad493558a6c13
Task vector (scale 1),dcaac641d40c05163b1ea0b736ba7f5ee89ec8a39ccb893b30489a59ddbd2645,164f355623c2da0028821e0942cad493558a6c13


## Held-out triplet results

SS is the component-macro stereotype preference score; 50 is the balance point.
A lower SS is not always better. LMS measures preference for meaningful sentences over
the unrelated sentence. ICAT combines LMS and balance. Scores are on a 0–100 scale;
changes are percentage points. Confidence intervals describe the paired change and use
1,000 component bootstrap replicates.

Inspect bias and LMS together. A change toward balanced preference can occur alongside
poorer sentence discrimination.

In [3]:
heldout = pd.concat(
    [
        pd.DataFrame(report["warbias_metric_values"]).assign(method=labels[method])
        for method, report in reports.items()
    ],
    ignore_index=True,
)
component_scope = (
    heldout["scope"].map(json.loads).map(lambda value: value == {"scope": "component_macro"})
)
triplets = heldout[
    heldout["task"].isin(["warbias_triplets_uk", "warbias_triplets_en"])
    & component_scope
    & heldout["metric"].isin(["ss", "lms", "icat"])
]
columns = ["method", "model", "metric", "before", "after", "change", "ci95", "uncertainty_status"]
for language in ("uk", "en"):
    print("Held-out triplets:", language)
    show(triplets[triplets["task"].eq("warbias_triplets_" + language)][columns])

Held-out triplets: uk


method,model,metric,before,after,change,ci95,uncertainty_status
WGA+KL,Qwen3 8B,icat,42.7992,42.9693,0.170113,"[-1.2032432866817173, 0.7871812386772872]",paired_cluster_percentile
WGA+KL,Qwen3 8B,lms,61.1786,60.881,-0.297619,"[-0.892857142857153, 0.0]",paired_cluster_percentile
WGA+KL,Qwen3 8B,ss,34.9789,35.2896,0.310706,"[0.0, 0.7801418439716272]",paired_cluster_percentile
WGA+KL,Llama 3.1 8B Instruct,icat,63.9397,57.8572,-6.08252,"[-9.284459176696943, 4.7372562595578716]",paired_cluster_percentile
WGA+KL,Llama 3.1 8B Instruct,lms,66.688,65.0917,-1.59632,"[-3.219696969696969, -0.3246753246753256]",paired_cluster_percentile
WGA+KL,Llama 3.1 8B Instruct,ss,47.9394,44.4428,-3.49659,"[-6.74334530717509, -1.1968085106383057]",paired_cluster_percentile
WGA+KL,Gemma 3 12B IT,icat,55.9771,58.7358,2.75873,"[-8.657731778179588, 6.064005219279793]",paired_cluster_percentile
WGA+KL,Gemma 3 12B IT,lms,65.2046,64.3002,-0.90437,"[-2.185952899051296, 0.1217532467532438]",paired_cluster_percentile
WGA+KL,Gemma 3 12B IT,ss,57.0758,54.3269,-2.74892,"[-5.523268398268403, -0.4761904761904745]",paired_cluster_percentile
WGA+KL,Lapa 12B Instruct,icat,41.3986,40.4712,-0.927364,"[-1.9995315294173215, 0.06984706263670566]",paired_cluster_percentile


Held-out triplets: en


method,model,metric,before,after,change,ci95,uncertainty_status
WGA+KL,Qwen3 8B,icat,30.9318,28.7548,-2.17701,"[-6.339122191682789, 1.0336887744566488]",paired_cluster_percentile
WGA+KL,Qwen3 8B,lms,47.0651,48.1099,1.04483,"[0.0, 2.532928064842956]",paired_cluster_percentile
WGA+KL,Qwen3 8B,ss,32.8607,29.8845,-2.97619,"[-8.035714285714285, 0.0]",paired_cluster_percentile
WGA+KL,Llama 3.1 8B Instruct,icat,51.8242,50.7748,-1.04939,"[-3.478723835908317, 4.36723039932566]",paired_cluster_percentile
WGA+KL,Llama 3.1 8B Instruct,lms,54.9589,55.0588,0.099878,"[-0.44959473150962737, 0.8980381320806927]",paired_cluster_percentile
WGA+KL,Llama 3.1 8B Instruct,ss,47.1482,46.1097,-1.0385,"[-3.5714285714285694, 0.455927051671722]",paired_cluster_percentile
WGA+KL,Gemma 3 12B IT,icat,44.2956,42.0219,-2.27372,"[-3.352024324136502, 1.5860150608907944]",paired_cluster_percentile
WGA+KL,Gemma 3 12B IT,lms,45.4033,44.2316,-1.17177,"[-2.2135632541217722, -0.30070605369806874]",paired_cluster_percentile
WGA+KL,Gemma 3 12B IT,ss,48.7802,47.5022,-1.27798,"[-2.880630008289586, 0.0]",paired_cluster_percentile
WGA+KL,Lapa 12B Instruct,icat,37.4053,37.2905,-0.114765,"[-0.9088289182055849, 0.5276608526553892]",paired_cluster_percentile


## QA results

Keep ambiguous and disambiguated accuracy separate. The saved bias scores use their
native denominator; an undefined score remains missing. Accuracy is a fraction (0–1),
while BBQ bias is on a signed −100 to 100 scale. Invalid answers remain separate.
The displayed QA scores weight semantic components equally.
The table shows the two 240-question QA views. Expanded and intersectional QA results
remain in the full export below.

In [4]:
qa = pd.DataFrame(qa_rows(reports))
qa["method"] = qa["method"].map(labels)
for language in ("uk", "en"):
    print("QA:", language)
    show(
        qa[
            qa["task"].eq("warbias_" + language)
            & qa["group"].isna()
            & qa["metric"].isin(["accuracy", "bias_score"])
        ][["method", "model", "condition", "metric", "before", "after", "change", "ci95"]]
    )

QA: uk


method,model,condition,metric,before,after,change,ci95
WGA+KL,Qwen3 8B,ambiguous,accuracy,0.8125,0.8125,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,ambiguous,bias_score,-8.75,-8.75,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_aligned,accuracy,1,1,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_aligned,bias_score,100,100,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_conflicting,accuracy,1,1,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_conflicting,bias_score,-100,-100,0,"[0.0, 0.0]"
WGA+KL,Llama 3.1 8B Instruct,ambiguous,accuracy,0.3375,0.3625,0.025,"[-0.012500000000000011, 0.07500000000000001]"
WGA+KL,Llama 3.1 8B Instruct,ambiguous,bias_score,-21.1436,-21.25,-0.106383,"[-4.631053536937549, 4.07232184336582]"
WGA+KL,Llama 3.1 8B Instruct,stereotype_aligned,accuracy,0.975,0.975,0,"[0.0, 0.0]"
WGA+KL,Llama 3.1 8B Instruct,stereotype_aligned,bias_score,95,95,0,"[0.0, 0.0]"


QA: en


method,model,condition,metric,before,after,change,ci95
WGA+KL,Qwen3 8B,ambiguous,accuracy,0.8375,0.8375,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,ambiguous,bias_score,-7.38636,-7.38636,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_aligned,accuracy,1,1,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_aligned,bias_score,100,100,0,"[0.0, 0.0]"
WGA+KL,Qwen3 8B,stereotype_conflicting,accuracy,1,0.9875,-0.0125,"[-0.03749999999999998, 0.0]"
WGA+KL,Qwen3 8B,stereotype_conflicting,bias_score,-100,-100,0,"[0.0, 0.0]"
WGA+KL,Llama 3.1 8B Instruct,ambiguous,accuracy,0.6,0.6,0,"[-0.050000000000000044, 0.050000000000000044]"
WGA+KL,Llama 3.1 8B Instruct,ambiguous,bias_score,11.4286,6.66667,-4.7619,"[-13.485846920289859, 1.349999999999997]"
WGA+KL,Llama 3.1 8B Instruct,stereotype_aligned,accuracy,0.9625,0.9625,0,"[-0.03749999999999998, 0.03749999999999998]"
WGA+KL,Llama 3.1 8B Instruct,stereotype_aligned,bias_score,100,100,0,"[0.0, 0.0]"


## Training-source diagnostic

These pairs evaluate the 137 training triplets. They show effects on the intervention's
own examples and do not measure held-out generalisation.

In [5]:
training_source = pd.concat(
    [
        pd.DataFrame(report["training_source_metric_values"]).assign(method=labels[method])
        for method, report in reports.items()
    ],
    ignore_index=True,
)
show(
    training_source[
        training_source["scope"].eq("component_macro")
        & training_source["metric"].isin(["ss", "lms", "icat"])
    ][["method", "model", "metric", "before", "after", "after_minus_before", "ci95"]]
)

method,model,metric,before,after,after_minus_before,ci95
WGA+KL,Qwen3 8B,ss,30.4053,30.2535,-0.151829,"[-1.9047619047619087, 1.4492753623188364]"
WGA+KL,Qwen3 8B,lms,67.6768,68.1819,0.505051,"[0.0, 1.5151515151515298]"
WGA+KL,Qwen3 8B,icat,41.1547,41.2548,0.100085,"[-2.8536040678897865, 2.3231188078299443]"
WGA+KL,Llama 3.1 8B Instruct,ss,42.9376,37.5723,-5.36525,"[-10.247401551749377, -0.9661835748792277]"
WGA+KL,Llama 3.1 8B Instruct,lms,79.9572,75.2782,-4.67893,"[-7.969336219336222, -1.5151515151515151]"
WGA+KL,Llama 3.1 8B Instruct,icat,68.6633,56.5675,-12.0958,"[-17.24692468940215, 16.19388240062346]"
WGA+KL,Gemma 3 12B IT,ss,27.0002,23.5346,-3.46561,"[-9.470899470899468, 0.0]"
WGA+KL,Gemma 3 12B IT,lms,75.2559,70.0531,-5.20275,"[-9.02360562143171, -1.7317742643829577]"
WGA+KL,Gemma 3 12B IT,icat,40.6385,32.9735,-7.66504,"[-17.774818194894273, -0.6724653946876157]"
WGA+KL,Lapa 12B Instruct,ss,59.6583,54.6153,-5.04298,"[-10.64935064935063, -0.9661835748792311]"


## Metric availability

Collection is complete, but some outcomes require more evidence. The saved 18-entry
inventory records each metric's status and reason. Severity needs source-bound reviewed
labels; benign-response quality needs calibrated judgments; broad capability retention
needs separate measurements. Undefined results are not zero effects.

This notebook uses the frozen native reports. Later exploratory NNR and demographic
analyses are separate from these reports and do not change their calibration status.

In [6]:
inventory = pd.concat(
    [
        pd.DataFrame(report["metric_status"]).assign(method=labels[method])
        for method, report in reports.items()
    ],
    ignore_index=True,
)
show(inventory.pivot(index=["method", "metric"], columns="model", values="status").reset_index())
show(inventory[["metric", "status", "reason"]].drop_duplicates().reset_index(drop=True))

model,method,metric,Gemma 3 12B IT,Lapa 12B Instruct,Llama 3.1 8B Instruct,Mamay 12B v2,Mamay 27B v2,Qwen3 8B
,Task vector (scale 1),BBQ,partial,available,partial,partial,partial,partial
,Task vector (scale 1),ICAT,available,available,available,available,available,available
,Task vector (scale 1),LMS,available,available,available,available,available,available
,Task vector (scale 1),M1,pending,pending,pending,pending,pending,pending
,Task vector (scale 1),M1b,pending,pending,pending,pending,pending,pending
,Task vector (scale 1),M2,pending,pending,pending,pending,pending,pending
,Task vector (scale 1),M3a,pending,pending,pending,pending,pending,pending
,Task vector (scale 1),M3b,pending,pending,pending,pending,pending,pending
,Task vector (scale 1),M3c,pending,pending,pending,pending,pending,pending
,Task vector (scale 1),M3d,partial,partial,partial,partial,partial,partial


metric,status,reason
forget_efficacy,available,None
NNR,pending,Frozen model/task/language neutrality calibration and eligible baseline cohort.
retain_utility,partial,In-scope native outcomes only; general utility and practical tolerances are not qualified.
LMS,available,None
ICAT,available,None
M1,pending,"One quality outcome, matched common content and an explicit independent-support rule."
M1b,pending,Qualified M1 panel and strictly positive maximum group quality.
M2,pending,Qualified common quality panel and paired group losses.
M3a,pending,Declared actor target and matched actor cells; no target is inferred.
M3b,pending,Declared reviewed topic target and reviewed near/far mapping.


## Export the complete tables

The export retains every saved scope, missing value and confidence interval.
Fair-UK owns scoring and uncertainty; this notebook does not recalculate formulas.
For new native runs, use the maintained `lm_eval.fair_uk` collector and
`paper compare-native` with the same source bundle and settings for both phases.

In [7]:
output = repo / "outputs/warbias_experiment/tables"
output.mkdir(parents=True, exist_ok=True)
for name, frame in {
    "heldout": heldout,
    "qa": qa,
    "training_source": training_source,
    "metric_inventory": inventory,
    "coverage": coverage,
}.items():
    frame.to_csv(output / (name + ".csv"), index=False)
    print(name + ".csv:", len(frame), "rows")
print("Saved tables: outputs/warbias_experiment/tables")

heldout.csv: 20352 rows


qa.csv: 22464 rows
training_source.csv: 1596 rows
metric_inventory.csv: 216 rows
coverage.csv: 144 rows
Saved tables: outputs/warbias_experiment/tables
